## 🔧 Cell 1: Check GPU & System Info

In [ ]:
# Check GPU availability
import subprocess

print("="*60)
print("   SYSTEM INFO")
print("="*60)

# GPU Check
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], 
                          capture_output=True, text=True)
    if result.returncode == 0:
        print("\n✅ GPU Available!")
        print(result.stdout)
    else:
        print("\n❌ GPU NOT DETECTED!")
        print("→ Go to Settings → Accelerator → Select GPU")
except:
    print("\n❌ nvidia-smi not found")

# Disk space
import shutil
total, used, free = shutil.disk_usage("/")
print(f"\n📁 Disk Space: {free // (2**30)} GB free / {total // (2**30)} GB total")

## 📦 Cell 2: Install Dependencies
This will take 3-5 minutes on first run

In [ ]:
%%time
print("="*60)
print("   INSTALLING DEPENDENCIES")
print("="*60)

# Core dependencies
!pip install -q insightface==0.7.3
!pip install -q onnxruntime-gpu==1.16.0
!pip install -q opencv-python-headless
!pip install -q tqdm

# GFPGAN for face enhancement
!pip install -q gfpgan

# CodeFormer (optional but recommended)
!pip install -q basicsr facexlib
!pip install -q git+https://github.com/sczhou/CodeFormer.git

# FFmpeg for audio processing
!apt-get -qq install ffmpeg

print("\n✅ All dependencies installed!")

## 🔍 Cell 3: Find Your Uploaded Files

In [ ]:
import os
import glob

print("="*60)
print("   FINDING YOUR FILES")
print("="*60)

# Kaggle input directory
INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

# Find all datasets
print("\n📂 Available datasets:")
if os.path.exists(INPUT_DIR):
    datasets = os.listdir(INPUT_DIR)
    for ds in datasets:
        print(f"   • {ds}")
        files = os.listdir(os.path.join(INPUT_DIR, ds))
        for f in files[:10]:  # Show first 10 files
            print(f"      └─ {f}")
else:
    print("   No datasets found!")

# Auto-find source image and target video
print("\n🔎 Searching for source image and target video...")

# Find images
images = glob.glob(f"{INPUT_DIR}/**/*.jpg", recursive=True) + \
         glob.glob(f"{INPUT_DIR}/**/*.jpeg", recursive=True) + \
         glob.glob(f"{INPUT_DIR}/**/*.png", recursive=True)

# Find videos
videos = glob.glob(f"{INPUT_DIR}/**/*.mp4", recursive=True) + \
         glob.glob(f"{INPUT_DIR}/**/*.avi", recursive=True) + \
         glob.glob(f"{INPUT_DIR}/**/*.mov", recursive=True)

print(f"\n📸 Found {len(images)} image(s):")
for img in images:
    print(f"   • {img}")

print(f"\n🎬 Found {len(videos)} video(s):")
for vid in videos:
    size_mb = os.path.getsize(vid) / (1024*1024)
    print(f"   • {vid} ({size_mb:.1f} MB)")

# Set default paths (modify these if auto-detection fails)
SOURCE_IMAGE = images[0] if images else None
TARGET_VIDEO = videos[0] if videos else None

print("\n" + "="*60)
if SOURCE_IMAGE and TARGET_VIDEO:
    print("✅ Files found! Using:")
    print(f"   Source: {SOURCE_IMAGE}")
    print(f"   Target: {TARGET_VIDEO}")
else:
    print("❌ Files not found! Please upload:")
    print("   1. A source face image (jpg/png)")
    print("   2. A target video (mp4/avi)")
print("="*60)

## ⚙️ Cell 4: Configuration
**Modify these settings as needed**

### 🎯 Enhancement Options Comparison

| Enhancement | Speed | 1 min video (30fps) | Quality | GPU Memory |
|------------|-------|---------------------|---------|------------|
| `none` | ~10-15 fps | ~2-3 min | Basic (may have artifacts) | Low (~4GB) |
| `gfpgan` | ~3-5 fps | ~10-15 min | Good (smooth faces) | Medium (~6GB) |
| `codeformer` | ~1-2 fps | ~20-30 min | Best (most realistic) | High (~8GB) |

### 💡 Recommendations:
- **Quick test**: Use `none` first to verify face detection works
- **Good balance**: Use `gfpgan` for most videos (recommended)
- **Best quality**: Use `codeformer` for final/important videos

In [ ]:
#============================================================
#   CONFIGURATION - MODIFY THESE PATHS IF NEEDED
#============================================================

# Source face image (the face you want to put in the video)
# SOURCE_IMAGE = "/kaggle/input/your-dataset/source.jpg"  # Uncomment and modify if auto-detection failed

# Target video (the video to swap faces in)
# TARGET_VIDEO = "/kaggle/input/your-dataset/target.mp4"  # Uncomment and modify if auto-detection failed

# Output settings
OUTPUT_VIDEO = "/kaggle/working/output_swapped.mp4"

#============================================================
#   🎯 ENHANCEMENT OPTIONS (Choose ONE by uncommenting)
#============================================================

# ───────────────────────────────────────────────────────────
# OPTION 1: NO ENHANCEMENT - FASTEST
# ───────────────────────────────────────────────────────────
# Speed:   ~10-15 fps
# Time:    ~2-3 minutes for 1 min video
# Quality: Basic swap, may have artifacts around face edges
# Use for: Quick testing, checking if face detection works
# ───────────────────────────────────────────────────────────
# ENHANCEMENT = "none"

# ───────────────────────────────────────────────────────────
# OPTION 2: GFPGAN - BALANCED ⭐ RECOMMENDED
# ───────────────────────────────────────────────────────────
# Speed:   ~3-5 fps  
# Time:    ~10-15 minutes for 1 min video
# Quality: Good restoration, smooth skin, fixes most artifacts
# Use for: Most videos, good speed/quality balance
# ───────────────────────────────────────────────────────────
ENHANCEMENT = "gfpgan"

# ───────────────────────────────────────────────────────────
# OPTION 3: CODEFORMER - BEST QUALITY
# ───────────────────────────────────────────────────────────
# Speed:   ~1-2 fps
# Time:    ~20-30 minutes for 1 min video  
# Quality: Highest quality, most realistic, preserves details
# Use for: Final renders, important videos, close-up faces
# ───────────────────────────────────────────────────────────
# ENHANCEMENT = "codeformer"

#============================================================
# Keep original audio?
KEEP_AUDIO = True
#============================================================

# Calculate and display estimated time
import cv2
if TARGET_VIDEO and os.path.exists(TARGET_VIDEO):
    cap = cv2.VideoCapture(TARGET_VIDEO)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / video_fps if video_fps > 0 else 0
    cap.release()
    
    fps_map = {"none": 12, "gfpgan": 4, "codeformer": 1.5}
    est_fps = fps_map.get(ENHANCEMENT, 4)
    est_time_min = frame_count / est_fps / 60
    est_time_str = f"~{est_time_min:.1f} minutes"
else:
    frame_count = "?"
    duration = "?"
    est_time_str = "Unknown (video not found)"

print("="*60)
print("   📋 CURRENT CONFIGURATION")
print("="*60)
print(f"\n   📸 Source Image: {SOURCE_IMAGE}")
print(f"   🎬 Target Video: {TARGET_VIDEO}")
print(f"   💾 Output Video: {OUTPUT_VIDEO}")
print(f"\n   ⚡ Enhancement:  {ENHANCEMENT.upper()}")
print(f"   🔊 Keep Audio:   {KEEP_AUDIO}")
if frame_count != "?":
    print(f"\n   📊 Video Info:")
    print(f"      • Frames: {frame_count}")
    print(f"      • Duration: {duration:.1f} seconds")
    print(f"   ⏱️  Est. Processing Time: {est_time_str}")
print("\n" + "="*60)

## 🎭 Cell 5: Video Face Swap Core Code

In [ ]:
import cv2
import numpy as np
import subprocess
import tempfile
import shutil
from typing import Optional, Tuple
from tqdm.notebook import tqdm
from enum import Enum

# Import face analysis
import insightface
from insightface.app import FaceAnalysis

# Import enhancement models
try:
    from gfpgan import GFPGANer
    GFPGAN_AVAILABLE = True
    print("✅ GFPGAN available")
except ImportError:
    GFPGAN_AVAILABLE = False
    print("⚠️ GFPGAN not available")

try:
    from codeformer.app import inference_app as codeformer_inference
    CODEFORMER_AVAILABLE = True
    print("✅ CodeFormer available")
except ImportError:
    CODEFORMER_AVAILABLE = False
    print("⚠️ CodeFormer not available")


class Enhancement(Enum):
    NONE = "none"
    GFPGAN = "gfpgan"
    CODEFORMER = "codeformer"


class VideoFaceSwap:
    """Video face swapping with enhancement for Kaggle."""
    
    def __init__(self, det_size: Tuple[int, int] = (640, 640)):
        self.det_size = det_size
        self.app = None
        self.swapper = None
        self.gfpgan = None
        self.codeformer = None
        
    def initialize(self) -> bool:
        """Initialize all models."""
        print("\n" + "="*60)
        print("   INITIALIZING MODELS")
        print("="*60)
        
        # Face Analyzer
        print("\n[1/4] Loading Face Analyzer (buffalo_l)...")
        try:
            self.app = FaceAnalysis(
                name='buffalo_l',
                providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
            )
            self.app.prepare(ctx_id=0, det_size=self.det_size)
            print("      ✅ Ready")
        except Exception as e:
            print(f"      ❌ Failed: {e}")
            return False
        
        # Face Swapper
        print("\n[2/4] Loading Face Swapper (inswapper_128)...")
        try:
            self.swapper = insightface.model_zoo.get_model(
                'inswapper_128.onnx', 
                download=True, 
                download_zip=True
            )
            print("      ✅ Ready")
        except Exception as e:
            print(f"      ❌ Failed: {e}")
            return False
        
        # GFPGAN
        print("\n[3/4] Loading GFPGAN...")
        if GFPGAN_AVAILABLE:
            try:
                self.gfpgan = GFPGANer(
                    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
                    upscale=1, 
                    arch='clean', 
                    channel_multiplier=2, 
                    bg_upsampler=None
                )
                print("      ✅ Ready")
            except Exception as e:
                print(f"      ⚠️ Failed: {e}")
        else:
            print("      ⚠️ Not available")
        
        # CodeFormer
        print("\n[4/4] Loading CodeFormer...")
        if CODEFORMER_AVAILABLE:
            self.codeformer = codeformer_inference
            print("      ✅ Ready")
        else:
            print("      ⚠️ Not available")
        
        print("\n" + "="*60)
        print("   ALL MODELS LOADED!")
        print("="*60)
        return True
    
    def enhance_frame(self, frame: np.ndarray, mode: Enhancement) -> np.ndarray:
        """Enhance a frame with GFPGAN or CodeFormer."""
        if mode == Enhancement.GFPGAN and self.gfpgan:
            try:
                _, _, result = self.gfpgan.enhance(
                    frame, has_aligned=False, 
                    only_center_face=False, 
                    paste_back=True
                )
                return result
            except:
                pass
        elif mode == Enhancement.CODEFORMER and self.codeformer:
            try:
                return self.codeformer(
                    image=frame, 
                    background_enhance=False, 
                    face_upsample=True, 
                    upscale=1, 
                    codeformer_fidelity=0.5
                )
            except:
                pass
        return frame
    
    def swap_frame(self, frame: np.ndarray, source_face, enhancement: Enhancement) -> np.ndarray:
        """Swap face in a single frame."""
        faces = self.app.get(frame)
        if not faces:
            return frame
        
        result = frame.copy()
        for face in faces:
            result = self.swapper.get(result, face, source_face, paste_back=True)
        
        if enhancement != Enhancement.NONE:
            result = self.enhance_frame(result, enhancement)
        
        return result
    
    def get_video_info(self, video_path: str) -> dict:
        """Get video metadata."""
        cap = cv2.VideoCapture(video_path)
        info = {
            "fps": cap.get(cv2.CAP_PROP_FPS),
            "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            "duration": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) / max(cap.get(cv2.CAP_PROP_FPS), 1)
        }
        cap.release()
        return info
    
    def extract_audio(self, video_path: str, audio_path: str) -> bool:
        """Extract audio from video using ffmpeg."""
        try:
            cmd = ["ffmpeg", "-y", "-i", video_path, "-vn", "-acodec", "copy", audio_path]
            subprocess.run(cmd, capture_output=True, check=True)
            return True
        except:
            return False
    
    def merge_audio_video(self, video_path: str, audio_path: str, output_path: str) -> bool:
        """Merge audio with video using ffmpeg."""
        try:
            cmd = [
                "ffmpeg", "-y",
                "-i", video_path,
                "-i", audio_path,
                "-c:v", "copy", "-c:a", "aac",
                "-strict", "experimental",
                output_path
            ]
            subprocess.run(cmd, capture_output=True, check=True)
            return True
        except:
            shutil.copy(video_path, output_path)
            return False
    
    def process_video(
        self,
        source_image: str,
        target_video: str,
        output_video: str,
        enhancement: Enhancement = Enhancement.GFPGAN,
        keep_audio: bool = True
    ) -> bool:
        """Process entire video."""
        print("\n" + "="*60)
        print("   VIDEO FACE SWAP - PROCESSING")
        print("="*60)
        
        # Validate inputs
        if not os.path.exists(source_image):
            print(f"❌ Source image not found: {source_image}")
            return False
        if not os.path.exists(target_video):
            print(f"❌ Target video not found: {target_video}")
            return False
        
        # Get source face
        print("\n[1/5] Analyzing source face...")
        source_img = cv2.imread(source_image)
        source_faces = self.app.get(source_img)
        if not source_faces:
            print("❌ No face found in source image!")
            return False
        source_face = max(source_faces, key=lambda x: x.det_score)
        print(f"      ✅ Found face ({source_face.det_score:.0%} confidence)")
        
        # Get video info
        print("\n[2/5] Reading video info...")
        info = self.get_video_info(target_video)
        print(f"      Resolution: {info['width']}x{info['height']}")
        print(f"      FPS: {info['fps']:.1f}")
        print(f"      Frames: {info['frame_count']}")
        print(f"      Duration: {info['duration']:.1f}s")
        
        # Estimate time
        fps_estimate = 3 if enhancement != Enhancement.NONE else 10  # frames per second
        est_time = info['frame_count'] / fps_estimate / 60
        print(f"      ⏱️ Estimated time: {est_time:.1f} minutes")
        
        # Temp directory
        temp_dir = tempfile.mkdtemp()
        temp_video = os.path.join(temp_dir, "temp_video.mp4")
        temp_audio = os.path.join(temp_dir, "temp_audio.aac")
        
        try:
            # Extract audio
            if keep_audio:
                print("\n[3/5] Extracting audio...")
                has_audio = self.extract_audio(target_video, temp_audio)
                print(f"      {'✅ Audio extracted' if has_audio else '⚠️ No audio found'}")
            else:
                has_audio = False
                print("\n[3/5] Skipping audio...")
            
            # Process frames
            print(f"\n[4/5] Processing {info['frame_count']} frames with {enhancement.value}...")
            cap = cv2.VideoCapture(target_video)
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(temp_video, fourcc, info['fps'], (info['width'], info['height']))
            
            pbar = tqdm(total=info['frame_count'], desc="Swapping faces", unit="frame")
            
            frame_count = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                swapped = self.swap_frame(frame, source_face, enhancement)
                out.write(swapped)
                pbar.update(1)
                frame_count += 1
            
            pbar.close()
            cap.release()
            out.release()
            print(f"      ✅ Processed {frame_count} frames")
            
            # Merge with audio
            print("\n[5/5] Finalizing video...")
            if has_audio and keep_audio:
                self.merge_audio_video(temp_video, temp_audio, output_video)
                print("      ✅ Audio merged")
            else:
                shutil.copy(temp_video, output_video)
            
            file_size = os.path.getsize(output_video) / (1024*1024)
            print(f"\n✅ Output saved: {output_video} ({file_size:.1f} MB)")
            
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
        
        print("\n" + "="*60)
        print("   🎉 COMPLETE!")
        print("="*60)
        return True


print("\n✅ VideoFaceSwap class loaded!")

## 🚀 Cell 6: Run Face Swap!

**Choose your run mode below:**
- **Single Mode**: Run with the enhancement you selected in Cell 4
- **Compare All Modes**: Run all 3 enhancements to compare quality (takes longer)

In [ ]:
%%time

#============================================================
#   RUN MODE - Choose ONE
#============================================================

# Option A: Single Enhancement (uses ENHANCEMENT from Cell 4)
RUN_MODE = "single"

# Option B: Compare ALL Enhancements (runs none, gfpgan, codeformer)
# RUN_MODE = "compare_all"

#============================================================

# Validate files exist
print("Checking files...")
if not os.path.exists(SOURCE_IMAGE):
    raise FileNotFoundError(f"Source image not found: {SOURCE_IMAGE}")
if not os.path.exists(TARGET_VIDEO):
    raise FileNotFoundError(f"Target video not found: {TARGET_VIDEO}")
print("✅ Files found!")

# Initialize
swapper = VideoFaceSwap(det_size=(640, 640))
if not swapper.initialize():
    raise RuntimeError("Failed to initialize models")

# Enhancement mapping
enhance_map = {
    "none": Enhancement.NONE,
    "gfpgan": Enhancement.GFPGAN,
    "codeformer": Enhancement.CODEFORMER
}

if RUN_MODE == "single":
    # ═══════════════════════════════════════════════════════════
    #   SINGLE MODE - Run with selected enhancement
    # ═══════════════════════════════════════════════════════════
    print(f"\n🎯 Running with {ENHANCEMENT.upper()} enhancement...")
    
    success = swapper.process_video(
        source_image=SOURCE_IMAGE,
        target_video=TARGET_VIDEO,
        output_video=OUTPUT_VIDEO,
        enhancement=enhance_map.get(ENHANCEMENT, Enhancement.GFPGAN),
        keep_audio=KEEP_AUDIO
    )
    
    if success:
        print("\n" + "🎉"*20)
        print("   VIDEO FACE SWAP COMPLETED!")
        print("🎉"*20)

elif RUN_MODE == "compare_all":
    # ═══════════════════════════════════════════════════════════
    #   COMPARE ALL MODE - Run all 3 enhancements
    # ═══════════════════════════════════════════════════════════
    print("\n" + "="*60)
    print("   🔄 RUNNING ALL 3 ENHANCEMENT MODES")
    print("="*60)
    
    modes = [
        ("none", "output_none.mp4", "⚡ NO ENHANCEMENT (Fastest)"),
        ("gfpgan", "output_gfpgan.mp4", "🔧 GFPGAN (Balanced)"),
        ("codeformer", "output_codeformer.mp4", "✨ CODEFORMER (Best Quality)")
    ]
    
    results = {}
    
    for mode_name, output_file, description in modes:
        print("\n" + "─"*60)
        print(f"   {description}")
        print("─"*60)
        
        output_path = f"/kaggle/working/{output_file}"
        
        import time
        start_time = time.time()
        
        success = swapper.process_video(
            source_image=SOURCE_IMAGE,
            target_video=TARGET_VIDEO,
            output_video=output_path,
            enhancement=enhance_map[mode_name],
            keep_audio=KEEP_AUDIO
        )
        
        elapsed = time.time() - start_time
        results[mode_name] = {
            "success": success,
            "time": elapsed,
            "output": output_path
        }
    
    # Summary
    print("\n" + "="*60)
    print("   📊 COMPARISON SUMMARY")
    print("="*60)
    print("\n   Mode         | Time      | Output File")
    print("   " + "-"*50)
    for mode_name, result in results.items():
        time_str = f"{result['time']/60:.1f} min"
        status = "✅" if result['success'] else "❌"
        print(f"   {status} {mode_name:<12} | {time_str:<9} | {result['output']}")
    
    print("\n" + "🎉"*20)
    print("   ALL MODES COMPLETED!")
    print("🎉"*20)
    print("\n   💡 Download all 3 videos from the Output tab to compare quality!")

## 👁️ Cell 7: Preview Result (Optional)

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode

# Display video in notebook (may not work for large videos)
if os.path.exists(OUTPUT_VIDEO):
    file_size = os.path.getsize(OUTPUT_VIDEO) / (1024*1024)
    
    if file_size < 50:  # Only embed if < 50MB
        print(f"Previewing: {OUTPUT_VIDEO} ({file_size:.1f} MB)")
        video_data = open(OUTPUT_VIDEO, "rb").read()
        video_b64 = b64encode(video_data).decode()
        display(HTML(f'''
        <video width="640" height="480" controls>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
        '''))
    else:
        print(f"Video too large to preview ({file_size:.1f} MB)")
        print(f"Download from: {OUTPUT_VIDEO}")
else:
    print("Output video not found!")

## 📥 Cell 8: Download Output

**Option 1: Direct Download**
- Go to **Output** tab (right sidebar)
- Click the file to download

**Option 2: Copy to Google Drive (if connected)**

In [ ]:
# List output files
print("📁 Output files:")
for f in os.listdir(WORKING_DIR):
    path = os.path.join(WORKING_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f"   • {f} ({size:.1f} MB)")

print("\n💡 To download: Click 'Output' tab on the right → Click on the file")

---

## 📚 Troubleshooting

### ❌ "No face found in source image"
- Use a clear, front-facing photo
- Ensure face is well-lit and visible
- Try a different source image

### ❌ "GPU not detected"
- Settings → Accelerator → GPU P100/T4
- Save and restart kernel

### ❌ "Out of memory"
- Use smaller video resolution
- Set `ENHANCEMENT = "none"`
- Restart kernel

### ❌ "Internet required"
- Settings → Internet → ON
- Verify phone number if prompted

### ⏱️ Taking too long?
- GFPGAN: ~3-5 fps
- CodeFormer: ~1-2 fps  
- No enhancement: ~10-15 fps
- Trim your video to shorter duration

---

## 🎯 Enhancement Mode Quick Reference

### Mode 1: NO ENHANCEMENT (`none`)
```
Speed:    ⚡⚡⚡⚡⚡ FASTEST (~10-15 fps)
Quality:  ⭐⭐ Basic  
Time:     ~2-3 min per 1 min video
GPU:      ~4GB VRAM
Best for: Quick tests, face detection check
```

### Mode 2: GFPGAN (`gfpgan`) ⭐ RECOMMENDED
```
Speed:    ⚡⚡⚡ MODERATE (~3-5 fps)
Quality:  ⭐⭐⭐⭐ Good
Time:     ~10-15 min per 1 min video  
GPU:      ~6GB VRAM
Best for: Most videos, balanced speed/quality
```

### Mode 3: CODEFORMER (`codeformer`)
```
Speed:    ⚡ SLOWEST (~1-2 fps)
Quality:  ⭐⭐⭐⭐⭐ Best
Time:     ~20-30 min per 1 min video
GPU:      ~8GB VRAM
Best for: Final renders, close-ups, important videos
```

---

## 💡 Tips

1. **Source Face Quality**: Higher quality = better results
2. **Similar Angles**: Works best when source & target faces have similar angles
3. **Lighting**: Similar lighting conditions improve blending
4. **Video Length**: Start with short clips (10-30 sec) to test
5. **Save Notebook**: Ctrl+S to save your configured notebook
6. **Compare Modes**: Use `RUN_MODE = "compare_all"` in Cell 6 to see all 3 results side by side

---

## 📋 Quick Steps Summary

1. ✅ Enable GPU (Settings → Accelerator → GPU)
2. ✅ Enable Internet (Settings → Internet → ON)
3. ✅ Upload files (Add Data → Upload → source.jpg + target.mp4)
4. ✅ Run Cell 1-5 in order
5. ✅ Choose enhancement in Cell 4
6. ✅ Run Cell 6 to process
7. ✅ Download from Output tab

---

**Happy Face Swapping! 🎭**